## Imports

In [1]:
import pandas as pd
import joblib

from pathlib import Path
from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedKFold,GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score,f1_score, roc_auc_score

## Data Loading & Splitting

In [2]:
df = pd.read_csv("../data/heart_disease.csv")

X = df.drop(columns="num")
y = (df["num"] > 0).astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2,random_state=42,stratify=y)

# Cross-validation 
cv = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

## Preprocessing Pipeline

In [3]:
numerical_cols = ["age", "trestbps", "chol", "thalach", "oldpeak", "ca"]
categorical_cols = ["cp", "restecg", "slope", "thal"]
binary_cols = ["sex", "fbs", "exang"]

preprocessing = ColumnTransformer([
    ("numerical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("scaler", StandardScaler())
    ]), numerical_cols),

    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(
            drop="first",
            handle_unknown="ignore",
            sparse_output=False
        ))
    ]), categorical_cols),

    ("binary", "passthrough", binary_cols)
])

## Models & Parameter Spaces

In [4]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "SVM": SVC(random_state=42)
}

parameters = {
    "Logistic Regression": {
        "model__C": [0.01, 0.1, 1, 10, 100]
    },
    "Decision Tree": {
        "model__max_depth": [None, 3, 5, 10],
        "model__min_samples_leaf": [1, 2, 4]
    },
    "Random Forest": {
        "model__n_estimators": [100, 200, 300],
        "model__max_depth": [None, 3, 5, 10],
        "model__min_samples_leaf": [1, 2, 4]
    },
    "SVM": {
        "model__C": [0.1, 1, 10, 100],
        "model__kernel": ["linear", "rbf"],
        "model__gamma": ["scale", "auto"]
    }
}

## Hyperparameter Tuning with Five-Fold Cross-Validation

In [5]:
searches = {}
baselines = {}
search_summary = []

for name, model in models.items():
    print("Tuning:", name)

    pipeline = Pipeline([
        ("preprocessing", clone(preprocessing)),
        ("selection", RFE(LogisticRegression(max_iter=2000, random_state=42),n_features_to_select=8)),
        ("model", clone(model))
    ])

    # Untuned version for the baseline comparison
    baselines[name] = clone(pipeline).fit(X_train, y_train)

    # Use GridSearch for small parameter spaces and RandomizedSearch for large ones to reduce computation.
    if name in ["Logistic Regression", "Decision Tree"]:
        search = GridSearchCV(pipeline,parameters[name],scoring="roc_auc",cv=cv,n_jobs=-1)
    else:
        search = RandomizedSearchCV(pipeline,parameters[name],n_iter=10,scoring="roc_auc",cv=cv,random_state=42,n_jobs=-1)

    search.fit(X_train, y_train)
    searches[name] = search

    search_summary.append({
        "Model": name,
        "Best_CV_AUC": search.best_score_,
        "Best_parameters": str(search.best_params_)
    })

tuning_df = pd.DataFrame(search_summary)
tuning_df.to_csv("../results/tuning_summary.csv", index=False)

tuning_df

Tuning: Logistic Regression
Tuning: Decision Tree
Tuning: Random Forest
Tuning: SVM


,Model,Best_CV_AUC,Best_parameters
0,Logistic Regression,0.907972,{'model__C': 0.1}
1,Decision Tree,0.866277,"{'model__max_depth': None, 'model__min_samples..."
2,Random Forest,0.906620,"{'model__n_estimators': 100, 'model__min_sampl..."
3,SVM,0.906717,"{'model__kernel': 'linear', 'model__gamma': 's..."


### Best Model Selection

In [6]:
best_name = max(searches,key=lambda name: searches[name].best_score_)

best_model = searches[best_name].best_estimator_

print("Selected model:", best_name)
print("Best parameters:", searches[best_name].best_params_)
print("Best CV AUC:", searches[best_name].best_score_)

Selected model: Logistic Regression
Best parameters: {'model__C': 0.1}
Best CV AUC: 0.9079718590588156


##  Baseline and Tuned Performance Comparison

In [7]:
results = []

for name in models:
    versions = {
        "Baseline": baselines[name],
        "Tuned": searches[name].best_estimator_
    }

    for version, model in versions.items():
        predictions = model.predict(X_test)

        if hasattr(model, "predict_proba"):
            scores = model.predict_proba(X_test)[:, 1]
        else:
            scores = model.decision_function(X_test)

        results.append({
            "Model": name,
            "Version": version,
            "Accuracy": accuracy_score(y_test, predictions),
            "Precision": precision_score(
                y_test, predictions, zero_division=0
            ),
            "Recall": recall_score(
                y_test, predictions, zero_division=0
            ),
            "F1": f1_score(y_test, predictions, zero_division=0),
            "ROC_AUC": roc_auc_score(y_test, scores)
        })

comparison_df = pd.DataFrame(results)
comparison_df.to_csv("../results/tuning_comparison.csv", index=False)

comparison_df.round(4)

,Model,Version,Accuracy,Precision,Recall,F1,ROC_AUC
0,Logistic Regression,Baseline,0.8361,0.8000,0.8571,0.8276,0.9318
1,Logistic Regression,Tuned,0.8689,0.8571,0.8571,0.8571,0.9297
2,Decision Tree,Baseline,0.8689,0.8571,0.8571,0.8571,0.8442
3,Decision Tree,Tuned,0.8852,0.8889,0.8571,0.8727,0.8452
4,Random Forest,Baseline,0.8689,0.8571,0.8571,0.8571,0.9210
5,Random Forest,Tuned,0.8525,0.8065,0.8929,0.8475,0.9491
6,SVM,Baseline,0.8689,0.8571,0.8571,0.8571,0.9361
7,SVM,Tuned,0.8689,0.8571,0.8571,0.8571,0.9297


### Final Model Export

In [8]:
joblib.dump(best_model, "../models/final_model.pkl")

['../models/final_model.pkl']

In [11]:
joblib.dump(best_model, "../models/final_model.pkl")

loaded_model = joblib.load("../models/final_model.pkl")

assert (loaded_model.predict(X_test) == best_model.predict(X_test)).all()

## Saving Results

In [10]:
with open("../results/evaluation_metrics.txt", "a") as file:
    file.write("\n\nHyperparameter Tuning\n")
    file.write(tuning_df.to_string(index=False))

    file.write("\n\nBaseline versus Tuned Test Metrics\n")
    file.write(comparison_df.round(4).to_string(index=False))

    file.write(f"\n\nSelected model: {best_name}\n")
    file.write("Exported pipeline: models/final_model.pkl\n")